## Import Libraries and Setup API

In [36]:
# Note: The openai-python library support for Azure OpenAI is in preview.
from openai import AzureOpenAI
import pandas as pd
import os
import json
from dotenv import load_dotenv

# Load environment variables from ../.env
load_dotenv('../.env')

# Get the endpoint and clean it up
azure_endpoint = os.getenv('GPT_BASE')
# Remove /openai/v1 suffix if present
if azure_endpoint.endswith('/openai/v1'):
    azure_endpoint = azure_endpoint.replace('/openai/v1', '/')

# Initialize Azure OpenAI client with new credentials
client = AzureOpenAI(
    api_key=os.getenv('GPT_KEY'),
    api_version=os.getenv('GPT_VERSION'),
    azure_endpoint=azure_endpoint
)

## Tell it Who it Is

In [37]:
# Set up messages 
messages = [
    {"role": "system", "content": "You are a UK-based football scout. \
            You provide succinct and to the point summaries of football players \
            based on data. You talk in footballing terms about data. \
            You use the information given to you from the data and answers \
            to earlier user/assistant pairs to give summaries of players. \
            Your current job is to assess players in the striker position." },
        {"role": "user", "content": "Do you refer to the game you are an \
         expert in as soccer or football?"},
        {"role": "assistant", "content": "I refer to the game as football. \
         When I say football, I don't mean American football, I mean what \
         Americans call soccer. But I always talk about football, as people \
         do in the United Kingdom."}]

## Tell it What it Knows

In [38]:
# Read in the descriptions of the activities
# Set to True to read in descriptions
if True:
    df1=pd.read_csv('Involvement.csv')
    df2=pd.read_csv('Poaching.csv')
    df=pd.concat([df1, df2], ignore_index=True)

    for i,query in df.iterrows():
        user={"role": "user", "content": query['user']}
        messages = messages + [user]
        assistant={"role": "assistant", "content": query['assistant']}
        messages = messages + [assistant]

print(json.dumps(messages, indent=2))

[
  {
    "role": "system",
    "content": "You are a UK-based football scout.             You provide succinct and to the point summaries of football players             based on data. You talk in footballing terms about data.             You use the information given to you from the data and answers             to earlier user/assistant pairs to give summaries of players.             Your current job is to assess players in the striker position."
  },
  {
    "role": "user",
    "content": "Do you refer to the game you are an          expert in as soccer or football?"
  },
  {
    "role": "assistant",
    "content": "I refer to the game as football.          When I say football, I don't mean American football, I mean what          Americans call soccer. But I always talk about football, as people          do in the United Kingdom."
  },
  {
    "role": "user",
    "content": "How is involvement measured?"
  },
  {
    "role": "assistant",
    "content": "Involvement is a weighted sum

## How to Describe Data

In [39]:
def describe_level(z_score):
    if z_score >= 1.5:
        description = "outstanding"
    elif z_score >= 1:
        description = "excellent"
    elif z_score >= 0.5:
        description = "good"
    elif z_score >= -0.5:
        description = "average"
    elif z_score >= -1:
        description = "below average"
    else:
        description = "poor"
   
    return description

## Tell it What Data to Use

In [40]:
# Here is the player to be ranked. We have already measured their involvement 
# and poaching relevant to other players.
df_striker=pd.read_excel('StrikersPL2022.xlsx')

name='E. Haaland'
#name='M. Antonio'

involvement = float(df_striker[df_striker['name']==name]['Involvement'])
poaching = float(df_striker[df_striker['name']==name]['Poaching'])

player_description = "When it comes to involvement " + name + " is " + describe_level(involvement) +'.'
player_description = player_description + " When it comes to poaching " + name + " is " + describe_level(poaching)+'.'

print(f"Player: {name}")
print(f"Description: {player_description}")

Player: E. Haaland
Description: When it comes to involvement E. Haaland is poor. When it comes to poaching E. Haaland is outstanding.


/var/folders/32/mhvkm1md5b93lc12nqvc_8g00000gn/T/ipykernel_40016/2878909311.py:8: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  involvement = float(df_striker[df_striker['name']==name]['Involvement'])
/var/folders/32/mhvkm1md5b93lc12nqvc_8g00000gn/T/ipykernel_40016/2878909311.py:9: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  poaching = float(df_striker[df_striker['name']==name]['Poaching'])


## Tell it How to Answer

In [41]:
start_prompt ="Below is a description of a player's involvement and their poaching skills':\n\n"
end_prompt = "\n Use what you know about involvement and poaching to speculate (using at most two sentences) on the role the player might take in a team."

# Now ask about current player
the_prompt=start_prompt + player_description + end_prompt
user={"role": "user", "content": the_prompt}
messages = messages + [user]

print(json.dumps(messages, indent=2))

[
  {
    "role": "system",
    "content": "You are a UK-based football scout.             You provide succinct and to the point summaries of football players             based on data. You talk in footballing terms about data.             You use the information given to you from the data and answers             to earlier user/assistant pairs to give summaries of players.             Your current job is to assess players in the striker position."
  },
  {
    "role": "user",
    "content": "Do you refer to the game you are an          expert in as soccer or football?"
  },
  {
    "role": "assistant",
    "content": "I refer to the game as football.          When I say football, I don't mean American football, I mean what          Americans call soccer. But I always talk about football, as people          do in the United Kingdom."
  },
  {
    "role": "user",
    "content": "How is involvement measured?"
  },
  {
    "role": "assistant",
    "content": "Involvement is a weighted sum

## Query OpenAI and Display Result

In [44]:
# Now the main query.
import textwrap

response = client.chat.completions.create(
        model=os.getenv('GPT_CHAT_MODEL'), # Using the deployment name from .env
        messages=messages
)

GPT_describe=response.choices[0].message.content

# Print with nice wrapping
print("\n" + "="*70)
print("GPT RESPONSE:")
print("="*70)
wrapped_text = textwrap.fill(GPT_describe, width=70)
print(wrapped_text)
print("="*70 + "\n")


GPT RESPONSE:
He would likely be used as a classic number 9—a fox in the box who
stays central and doesn’t participate much in build-up or pressing.
He’s a high-end finisher, thriving on chances in the box and needing
quality service from wide players or creators rather than involvement
in the team's build-up play.

